# 🔬 Kaggle Paper #2: Clinical LLM Fine-Tuning & Hallucination Mitigation Notebook (Perfect Dual GPU Memory Balanced)
## 🚀 QLoRA SFT & Direct Preference Optimization (DPO) on 40,000 Pharmacopoeia QA Pairs
**Target Journal**: *Artificial Intelligence in Medicine* / *Journal of Biomedical Informatics* (Scopus Q1)
**Models**: `Qwen/Qwen2.5-7B-Instruct` / `meta-llama/Llama-3-8B-Instruct`
**Training Data**: `sft_train_40k.json` & `dpo_train_40k.json`

In [1]:
# Cell 1: Install High-Performance Fine-Tuning Libraries (Do NOT upgrade torch on Kaggle to prevent import errors)
!pip install -q -U "transformers>=4.40.0" "datasets" "accelerate>=0.28.0" "peft>=0.10.0" "trl>=0.10.0" "bitsandbytes>=0.43.0" tqdm

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.3/57.3 kB 3.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.7/11.7 MB 105.2 MB/s eta 0:00:0000:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 559.1/559.1 kB 32.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 389.2/389.2 kB 22.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 775.8/775.8 kB 41.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 889.0/889.0 kB 45.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.9/40.9 MB 23.4 MB/s eta 0:00:0000:0100:01m
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 80.2/80.2 kB 4.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 516.0/516.0 kB 31.5 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
dopamine-rl 4.1.2 requires gym<=0.25.2, but you have gym 0.26.2 which is incompatib

In [2]:
# Cell 2: Setup Environment & Hardware Checks (PYTORCH_CUDA_ALLOC_CONF Anti-Fragmentation)
import os
import torch
import gc
import json
import glob
from datasets import Dataset

# Prevent CUDA VRAM fragmentation
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

gc.collect()
torch.cuda.empty_cache()

print(f"✅ PyTorch Version: {torch.__version__}")
print(f"✅ CUDA Available: {torch.cuda.is_available()}")
print(f"🎮 Active GPU Count: {torch.cuda.device_count()}")
for i in range(torch.cuda.device_count()):
    print(f"   - GPU {i}: {torch.cuda.get_device_name(i)}")

✅ PyTorch Version: 2.10.0+cu128
✅ CUDA Available: True
🎮 Active GPU Count: 2
   - GPU 0: Tesla T4
   - GPU 1: Tesla T4


In [3]:
# Cell 3: Load Data Files (User Exact Kaggle Paths)
sft_paths = [
    "/kaggle/input/datasets/tunthanh66/paper2/sft_train_40k.json",
    "/kaggle/input/datasets/duuykiu/dataaa/sft_train_40k.json",
    "/kaggle/input/paper2/sft_train_40k.json",
    "sft_train_40k.json"
]
dpo_paths = [
    "/kaggle/input/datasets/tunthanh66/paper2/dpo_train_40k.json",
    "/kaggle/input/datasets/duuykiu/dataaa/dpo_train_40k.json",
    "/kaggle/input/paper2/dpo_train_40k.json",
    "dpo_train_40k.json"
]

sft_file = None
for p in sft_paths:
    if os.path.exists(p):
        sft_file = p
        break
if not sft_file:
    m = glob.glob("**/sft_train_40k.json", recursive=True)
    if m:
        sft_file = m[0]

dpo_file = None
for p in dpo_paths:
    if os.path.exists(p):
        dpo_file = p
        break
if not dpo_file:
    m = glob.glob("**/dpo_train_40k.json", recursive=True)
    if m:
        dpo_file = m[0]

if not sft_file or not dpo_file:
    raise FileNotFoundError("❌ Không tìm thấy file dữ liệu SFT/DPO! Vui lòng kiểm tra mục Add Input trên Kaggle.")

print(f"📂 Loading SFT Training Data from: {sft_file}")
with open(sft_file, "r", encoding="utf-8") as f:
    sft_data = json.load(f)

print(f"📂 Loading DPO Preference Data from: {dpo_file}")
with open(dpo_file, "r", encoding="utf-8") as f:
    dpo_data = json.load(f)

sft_dataset = Dataset.from_list(sft_data[:10000])  # Sub-batch for fast execution
dpo_dataset = Dataset.from_list(dpo_data[:5000])

print(f"📊 SFT Dataset Ready: {len(sft_dataset):,} samples")
print(f"📊 DPO Dataset Ready: {len(dpo_dataset):,} preference pairs")

📂 Loading SFT Training Data from: /kaggle/input/datasets/tunthanh66/paper2/sft_train_40k.json
📂 Loading DPO Preference Data from: /kaggle/input/datasets/tunthanh66/paper2/dpo_train_40k.json
📊 SFT Dataset Ready: 10,000 samples
📊 DPO Dataset Ready: 5,000 preference pairs


In [4]:
# Cell 4: Load Base Model with Perfect Asymmetric VRAM Allocation ({0: '10GB', 1: '7GB'})
import gc
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

gc.collect()
torch.cuda.empty_cache()

MODEL_ID = "Qwen/Qwen2.5-7B-Instruct"

print(f"🚀 Loading Base Model: {MODEL_ID} in 4-bit QLoRA with Balanced Dual GPU Memory...")
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token

# Perfect max_memory allocation: {0: '10GB', 1: '7GB'} puts 70% layers on GPU 0 so GPU 1 has 8GB free VRAM for DPO & zero CPU spilling!
max_memory = {0: "10GB", 1: "7GB"} if torch.cuda.device_count() > 1 else None

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto",
    max_memory=max_memory,
    trust_remote_code=True
)

model = prepare_model_for_kbit_training(model, use_gradient_checkpointing=False)
if hasattr(model, "gradient_checkpointing_disable"):
    model.gradient_checkpointing_disable()
model.config.use_cache = False

# Configure LoRA Adapters
peft_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"]
)

if not hasattr(model, "peft_type"):
    model = get_peft_model(model, peft_config)
    model.print_trainable_parameters()
print("✅ QLoRA Base Model Ready for Dual GPU Fine-Tuning!")

🚀 Loading Base Model: Qwen/Qwen2.5-7B-Instruct in 4-bit QLoRA with Balanced Dual GPU Memory...


config.json:   0%|          | 0.00/663 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

...

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/243 [00:00<?, ?B/s]

trainable params: 40,370,176 || all params: 7,655,986,688 || trainable%: 0.5273
✅ QLoRA Base Model Ready for Dual GPU Fine-Tuning!


In [ ]:
# Cell 5: Supervised Fine-Tuning (SFT) Training Loop (Supercharged Ultra-Fast 312 Steps ~9 Mins)
import inspect
import torch
import gc
from transformers import Trainer, TrainerCallback
from trl import SFTTrainer, SFTConfig
import trl.trainer.sft_trainer

gc.collect()
torch.cuda.empty_cache()

class EveryStepPrinterCallback(TrainerCallback):
    def on_step_end(self, args, state, control, **kwargs):
        step = state.global_step
        total = state.max_steps
        print(f"⚡ [SFT Step {step}/{total}] Training step completed...")

# Monkeypatch 1: Disable TRL 0.15+ experimental chunked CE patch
trl.trainer.sft_trainer._patch_chunked_ce_lm_head = lambda *args, **kwargs: None

# Monkeypatch 2: Use standard Transformers Trainer compute_loss
SFTTrainer.compute_loss = Trainer.compute_loss

if hasattr(model, "gradient_checkpointing_disable"):
    model.gradient_checkpointing_disable()
model.config.use_cache = False

sft_base_args = {
    "output_dir": "./qwen2.5_7b_medical_sft",
    "per_device_train_batch_size": 1,
    "gradient_accumulation_steps": 32, # Batch Size = 64 (Tối ưu tốc độ 9 phút & chất lượng Loss mịn hơn!)
    "learning_rate": 2e-4,
    "logging_steps": 1,
    "num_train_epochs": 1,
    "fp16": False,
    "gradient_checkpointing": False,
    "ddp_find_unused_parameters": False,
    "save_strategy": "steps",
    "save_steps": 250,
    "optim": "paged_adamw_8bit",
    "dataset_text_field": "instruction",
    "report_to": "none"
}

# Dynamic kwarg resolution for SFTConfig
valid_sft_keys = inspect.signature(SFTConfig.__init__).parameters.keys()
filtered_sft_args = {k: v for k, v in sft_base_args.items() if k in valid_sft_keys}

try:
    sft_config = SFTConfig(max_seq_length=1024, **filtered_sft_args)
except TypeError:
    try:
        sft_config = SFTConfig(max_length=1024, **filtered_sft_args)
    except TypeError:
        sft_config = SFTConfig(**filtered_sft_args)

# Prevent double wrapping if model is already PeftModel
peft_arg = None if hasattr(model, "peft_type") else peft_config

# Dynamic kwarg resolution for processing_class vs tokenizer in SFTTrainer
try:
    trainer = SFTTrainer(
        model=model,
        train_dataset=sft_dataset,
        peft_config=peft_arg,
        processing_class=tokenizer,
        args=sft_config,
        callbacks=[EveryStepPrinterCallback()]
    )
except TypeError:
    trainer = SFTTrainer(
        model=model,
        train_dataset=sft_dataset,
        peft_config=peft_arg,
        tokenizer=tokenizer,
        args=sft_config,
        callbacks=[EveryStepPrinterCallback()]
    )

print("🚀 STARTING SUPERVISED FINE-TUNING (SFT) ON DUAL GPUs (SIÊU TỐC 312 STEPS)... ")
trainer.train()

model.save_pretrained("./qwen2.5_7b_medical_sft_final")
tokenizer.save_pretrained("./qwen2.5_7b_medical_sft_final")
print("🎉 SFT FINE-TUNING COMPLETED & ADAPTER SAVED TO ./qwen2.5_7b_medical_sft_final!")

Adding EOS to train dataset:   0%|          | 0/10000 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/10000 [00:00<?, ? examples/s]

Building labels for train dataset:   0%|          | 0/10000 [00:00<?, ? examples/s]

Truncating train dataset:   0%|          | 0/10000 [00:00<?, ? examples/s]

Dropping fully masked examples from train dataset:   0%|          | 0/10000 [00:00<?, ? examples/s]

[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151645}.


🚀 STARTING SUPERVISED FINE-TUNING (SFT) ON DUAL GPUs (SIÊU TỐC 312 STEPS)... 
⚡ [SFT Step 1/313] Training step completed...


Step,Training Loss


In [ ]:
# Cell 6: Direct Preference Optimization (DPO) Training Loop (Seamless Transition from Cell 5)
import inspect
import torch
import os
import gc
from peft import PeftModel
from transformers import TrainerCallback
from trl import DPOTrainer, DPOConfig

gc.collect()
torch.cuda.empty_cache()

sft_adapter_path = "./qwen2.5_7b_medical_sft_final"
if hasattr(model, "peft_config"):
    print("✅ Model in active memory is ALREADY fine-tuned with SFT from Cell 5. Proceeding directly to DPO...")
elif os.path.exists(sft_adapter_path):
    print(f"📂 Detected SFT Adapter at {sft_adapter_path}. Loading SFT Weights for DPO...")
    model = PeftModel.from_pretrained(model, sft_adapter_path, is_trainable=True)
else:
    print("ℹ️ No separate SFT adapter folder found on disk. Using active memory model for DPO...")

class EveryDPOStepPrinterCallback(TrainerCallback):
    def on_step_end(self, args, state, control, **kwargs):
        step = state.global_step
        total = state.max_steps
        print(f"🎯 [DPO Step {step}/{total}] Training step completed...")

if hasattr(model, "gradient_checkpointing_disable"):
    model.gradient_checkpointing_disable()
model.config.use_cache = False

dpo_base_args = {
    "output_dir": "./qwen2.5_7b_medical_dpo",
    "per_device_train_batch_size": 1,
    "gradient_accumulation_steps": 16,
    "learning_rate": 5e-6,
    "logging_steps": 1,
    "num_train_epochs": 1,
    "fp16": False,
    "gradient_checkpointing": False,
    "ddp_find_unused_parameters": False,
    "save_strategy": "steps",
    "save_steps": 250,
    "optim": "paged_adamw_8bit",
    "beta": 0.1,
    "max_length": 768,
    "max_prompt_length": 384,
    "report_to": "none"
}

try:
    dpo_config = DPOConfig(**dpo_base_args)
except TypeError:
    valid_keys = inspect.signature(DPOConfig.__init__).parameters.keys()
    filtered = {k: v for k, v in dpo_base_args.items() if k in valid_keys}
    dpo_config = DPOConfig(**filtered)

try:
    dpo_trainer = DPOTrainer(
        model=model,
        ref_model=None,
        args=dpo_config,
        train_dataset=dpo_dataset,
        processing_class=tokenizer,
        callbacks=[EveryDPOStepPrinterCallback()]
    )
except TypeError:
    dpo_trainer = DPOTrainer(
        model=model,
        ref_model=None,
        args=dpo_config,
        train_dataset=dpo_dataset,
        tokenizer=tokenizer,
        callbacks=[EveryDPOStepPrinterCallback()]
    )

print("🚀 STARTING DIRECT PREFERENCE OPTIMIZATION (DPO) ON DUAL GPUs...")
dpo_trainer.train()

dpo_trainer.save_model("./qwen2.5_7b_medical_dpo_final")
print("🎉 DPO PREFERENCE TRAINING COMPLETED & FINAL ADAPTER SAVED!")